Activation:
A good model need to work well even in a robust circumstance.

所以我们在训练模型的时候,可以在层之间加入噪声,使得模型更加具有鲁棒性.

dropout就是在层间无偏差的加入噪声,使得模型更加健壮.具体而言,我们希望新的层间输入$\mathbf{x}'$期望不变,即$E[\mathbf{x}'] = \mathbf{x}$.故我们使以下等式成立:
$$
x_i'=\begin{cases}
\dfrac{x_i}{1-p} & \text{with probability } p \\
0 & \text{otherwise}
\end{cases}
$$

在神经网络中,我们一般通过随机的将层间的节点置为0来实现上述操作(以上操作均是在训练过程中进行的,推理的过程中不能使用dropout这会使模型的输出结果不稳定).



In [ ]:
import torch
import numpy as np
import torch.nn as nn
from torchvision import transforms
from torch.utils import data
import torchvision

In [ ]:
def dropout_layer(X,dropout):
    assert 0.0<=dropout<=1.0
    if dropout==1.0:
        return torch.zeros_like(X)
    if dropout==0.0:
        return X
    else:
        mask=(torch.rand(X.shape)>dropout).float()
        return mask*X/(1-dropout)

In [ ]:
X= torch.arange(16, dtype = torch.float32).reshape((2, 8))
print(torch.rand(X.shape))
print(dropout_layer(X,0.5))
print(dropout_layer(X,0.1))

In [ ]:
num_inputs=784
num_outputs=10
num_hidden_1=256
num_hidden_2=256

In [ ]:
dropout1=0.2
dropout2=0.8
class Net(nn.Module):
    def __init__(self,num_inputs,num_outputs,num_hidden_1,num_hidden_2,is_training=True):
        super(Net,self).__init__()
        self.num_inputs=num_inputs
        self.num_outputs=num_outputs
        self.training=is_training
        self.lin1=nn.Linear(num_inputs,num_hidden_1)
        self.lin2=nn.Linear(num_hidden_1,num_hidden_2)
        self.lin3=nn.Linear(num_hidden_2,num_outputs)
        self.relu=nn.ReLU()
    
    def forward(self,X):
        H1=self.relu(self.lin1(X.reshape(-1,self.num_inputs)))
        if self.training==True:
            H1=dropout_layer(H1,dropout1)
        H2=self.relu(self.lin2(H1))
        if self.training==True:
            H2=dropout_layer(H2,dropout2)
        out=self.lin3(H2)
        return out

net=Net(num_inputs,num_outputs,num_hidden_1,num_hidden_2)

In [ ]:
num_epoch=10
loss=nn.CrossEntropyLoss(reduction='none')
trainer=torch.optim.Adam(net.parameters(),lr=0.001)


In [ ]:

mnist_train=torchvision.datasets.MNIST(root='./data',train=True,transform=transforms.ToTensor(),download=True)
mnist_test=torchvision.datasets.MNIST(root='./data',train=False,transform=transforms.ToTensor(),download=True)
train_iter=data.DataLoader(mnist_train,batch_size=256,shuffle=True)
test_iter=data.DataLoader(mnist_test,batch_size=256,shuffle=False)

In [ ]:
from utils.tool import train_classify

train_classify(net,train_iter,test_iter,loss,trainer,num_epoch)


In [ ]:
from utils.tool import evaluate_accuracy
evaluate_accuracy(net,test_iter)

In [ ]:
net=nn.Sequential(nn.Flatten(),nn.Linear(784,256),nn.ReLU(),nn.Dropout(dropout1),
                  nn.Linear(256,256),nn.ReLU(),nn.Dropout(dropout2),
                  nn.Linear(256,10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)
net.apply(init_weights)


In [ ]:
train_classify(net,train_iter,test_iter,loss,trainer,num_epoch)